In [ ]:
import pandas as pd
from google.cloud import bigquery
from lightgbm import LGBMRegressor

client = bigquery.Client(project="prc-data-challenge-2025")


def q(sql):
    return client.query(sql).to_dataframe()

In [ ]:
df = q(
    """ 
SELECT ft.idx, ft.fuel_kg, ft.flight_id, flt.aircraft_type, ((ft.`end` - ft.start)/1e9) as duration_s, 
TIMESTAMP_DIFF(TIMESTAMP_MICROS(DIV(ft.start, 1000)), flt.takeoff, SECOND) AS since_takeoff_s,
TIMESTAMP_DIFF(flt.landed, TIMESTAMP_MICROS(DIV(ft.`end`, 1000)), SECOND) AS before_landing_s
FROM `prc-data-challenge-2025.prc2025.fuel_train` as ft
JOIN `prc-data-challenge-2025.prc2025.flightlist_train` as flt
ON ft.flight_id = flt.flight_id
"""
)

In [ ]:
df.describe()

131,530 rows, nothing missing, so the join didn't lose or duplicate any interval.
Some intervals start up to 4 h before takeoff or end almost 3 h after landing, which can't be right. Same for the max fuel_kg: 32 t in less than an hour.

In [ ]:
flights = df.groupby("flight_id").agg(
    fuel_kg_sum=("fuel_kg", "sum"),
    duration_s_sum=("duration_s", "sum"),
    aircraft_type=("aircraft_type", "first"),
)

flights["fuel_rate_kg_per_s"] = flights["fuel_kg_sum"] / flights["duration_s_sum"]

One row per flight with its average fuel rate, to run the Tukey rule like in explore_links.

In [ ]:
# Tukey far-out bounds per aircraft type and outlier flights
grouped = flights.groupby("aircraft_type")["fuel_rate_kg_per_s"]
flights["q1"] = grouped.transform(lambda s: s.quantile(0.25))
flights["q3"] = grouped.transform(lambda s: s.quantile(0.75))
flights["iqr"] = flights["q3"] - flights["q1"]
flights["lower_bound"] = flights["q1"] - 3 * flights["iqr"]
flights["upper_bound"] = flights["q3"] + 3 * flights["iqr"]

outliers = flights[
    (flights["fuel_rate_kg_per_s"] < flights["lower_bound"])
    | (flights["fuel_rate_kg_per_s"] > flights["upper_bound"])
]
len(outliers)

205 flights, same as in explore_links.

In [ ]:
ground_overflow = df[(df["since_takeoff_s"] < -1800) | (df["before_landing_s"] < -1800)]
ground_flights = ground_overflow["flight_id"].unique()
len(ground_flights)

189 flights (410 intervals) go more than 30 min outside the flight.

In [ ]:
bad_flights = set(outliers.index) | set(ground_flights)
df_train = df[~df["flight_id"].isin(bad_flights)]
len(df_train), len(df)

385 flights to remove in total, only 9 are in both lists. That's 3.5% of the flights and 3% of the intervals.

In [ ]:
flights = df_train["flight_id"].unique()
val_flights = pd.Series(flights).sample(frac=0.2, random_state=42)

val = df_train[df_train["flight_id"].isin(val_flights)]
fit = df_train[~df_train["flight_id"].isin(val_flights)]

In [ ]:
fit_rate = fit.fuel_kg / fit.duration_s
rate_by_type = fit_rate.groupby(fit.aircraft_type).median()

val["rate"] = val.aircraft_type.map(rate_by_type).fillna(fit_rate.median())
val["fuel_pred"] = val["rate"] * val["duration_s"]
val["sq_error"] = (val["fuel_kg"] - val["fuel_pred"]) ** 2
rmse = (val["sq_error"].mean()) ** 0.5
print(rmse)

First baseline: 213 kg. Too good to be true, the outliers were also removed from val.

In [ ]:
flights = df["flight_id"].unique()
val_flights = pd.Series(flights).sample(frac=0.2, random_state=42)

val = df[df["flight_id"].isin(val_flights)]
fit = df[(~df["flight_id"].isin(val_flights)) & (~df["flight_id"].isin(bad_flights))]

fit_rate = fit.fuel_kg / fit.duration_s
rate_by_type = fit_rate.groupby(fit.aircraft_type).median()

val["rate"] = val.aircraft_type.map(rate_by_type).fillna(fit_rate.median())
val["fuel_pred"] = val["rate"] * val["duration_s"]
val["sq_error"] = (val["fuel_kg"] - val["fuel_pred"]) ** 2
rmse = (val["sq_error"].mean()) ** 0.5
print(rmse)

With outliers kept in val (removed from fit only) the baseline gets 330 kg. This is the score to beat.
Training without the cleaning gives 329.9, basically the same since the median ignores extreme values anyway.

In [ ]:
features = ["duration_s", "since_takeoff_s", "before_landing_s", "aircraft_type"]

X_fit = fit[features]
X_fit["aircraft_type"] = X_fit["aircraft_type"].astype("category")
y_fit = fit["fuel_kg"]

X_val = val[features]
X_val["aircraft_type"] = X_val["aircraft_type"].astype("category")
y_val = val["fuel_kg"]


model = LGBMRegressor(random_state=42)
model.fit(X_fit, y_fit)
val_pred_lgbm = model.predict(X_val)

rmse_lgbm = ((y_val - val_pred_lgbm) ** 2).mean() ** 0.5
print(rmse_lgbm)

LightGBM with default parameters on the 4 features: 299 kg, about 9% better than the baseline.
Part of what's left comes from the outlier flights in val, and no model can guess those.

In [ ]:
val = df[df["flight_id"].isin(val_flights)]
fit = df[~df["flight_id"].isin(val_flights)]

features = ["duration_s", "since_takeoff_s", "before_landing_s", "aircraft_type"]

X_fit = fit[features]
X_fit["aircraft_type"] = X_fit["aircraft_type"].astype("category")
y_fit = fit["fuel_kg"]

X_val = val[features]
X_val["aircraft_type"] = X_val["aircraft_type"].astype("category")
y_val = val["fuel_kg"]


model = LGBMRegressor(random_state=42)
model.fit(X_fit, y_fit)
val_pred_lgbm = model.predict(X_val)

rmse_lgbm = ((y_val - val_pred_lgbm) ** 2).mean() ** 0.5
print(rmse_lgbm)

Same model but trained with the outliers: 305 kg, so worse. Unlike the median, LightGBM gets pulled by the extreme values. We keep the cleaning.

## Baseline on rank and final

The 330 kg above was on the old random val, not comparable with the final score (221.6 on rank + final, run #26). Same baseline with the final protocol: median rates by type from all of train cleaned with clean_training, scored on rank, final and both together.

In [ ]:
from prc_fuel.clean import clean_training
from prc_fuel.evaluate import rmse
from prc_fuel.features import duration_s
from prc_fuel.load import load

train_all = clean_training(load("train"))
rate = train_all["fuel_kg"] / duration_s(train_all)
rate_by_type = rate.groupby(train_all["aircraft_type"]).median()


def baseline(df):
    df_rate = df["aircraft_type"].map(rate_by_type).fillna(rate.median())
    return df_rate * duration_s(df)


rank = load("rank")
final = load("final")
sets = {
    "rank": rank,
    "final": final,
    "rank + final": pd.concat([rank, final], ignore_index=True),
}

pd.Series({name: rmse(d["fuel_kg"], baseline(d)) for name, d in sets.items()})

Baseline with the final protocol: 266.6 on rank, 249.3 on final, 256.3 on rank + final.
Final model (run #26): 233.2 / 213.7 / 221.6, so -34.7 kg on rank + final (-13.5%), about the same gain on rank and on final.
The gap probably looks small partly because the outlier flights add the same error to both scores, neither can predict them.